In [15]:
import os
import pandas as pd
import numpy as np

In [16]:
mhs = pd.read_csv("../data/processed/mhs_main_experiment_annotations.csv")

print("Loaded shape:", mhs.shape)
print("Unique comments:", mhs["comment_id"].nunique())
print("Unique annotators:", mhs["annotator_id"].nunique())

Loaded shape: (49433, 52)
Unique comments: 19761
Unique annotators: 7851


In [17]:
target_summary = (
    mhs.groupby("target_type")
    .agg(
        annotation_rows=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        unique_annotators=("annotator_id", "nunique")
    )
    .reset_index()
    .sort_values("annotation_rows", ascending=False)
)

target_summary

,target_type,annotation_rows,unique_comments,unique_annotators
2,women_only,26905,11596,7377
0,immigrant_only,21544,8222,7134
1,women_and_immigrant,984,609,842


In [18]:
survey_item_cols = [
    "sentiment",
    "respect",
    "insult",
    "humiliate",
    "status",
    "dehumanize",
    "violence",
    "genocide",
    "attack_defend",
    "hatespeech"
]

In [19]:
target_summary = (
    mhs.groupby("target_type")
    .agg(
        annotation_rows=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        unique_annotators=("annotator_id", "nunique")
    )
    .sort_values("annotation_rows", ascending=False)
)

women_gender_summary = (
    mhs[mhs["is_women_targeted"] == 1]
    .groupby("annotator_gender_group")
    .agg(
        annotation_rows=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        unique_annotators=("annotator_id", "nunique")
    )
    .sort_values("annotation_rows", ascending=False)
)

immigrant_ideology_summary = (
    mhs[mhs["is_immigrant_targeted"] == 1]
    .groupby("annotator_ideology_group")
    .agg(
        annotation_rows=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        unique_annotators=("annotator_id", "nunique")
    )
    .sort_values("annotation_rows", ascending=False)
)

In [20]:
comment_level = (
    mhs.groupby("comment_id")
    .agg(
        target_type=("target_type", "first"),
        is_women_targeted=("is_women_targeted", "max"),
        is_immigrant_targeted=("is_immigrant_targeted", "max"),
        annotation_count=("annotator_id", "count"),
        unique_annotators=("annotator_id", "nunique"),
        mean_hatespeech=("hatespeech", "mean"),
        mean_sentiment=("sentiment", "mean"),
        mean_respect=("respect", "mean"),
        mean_insult=("insult", "mean"),
        mean_humiliate=("humiliate", "mean"),
        mean_status=("status", "mean"),
        mean_dehumanize=("dehumanize", "mean"),
        mean_violence=("violence", "mean"),
        mean_genocide=("genocide", "mean"),
        mean_attack_defend=("attack_defend", "mean")
    )
    .reset_index()
)

In [21]:
# Safe hatespeech severity buckets

try:
    comment_level["hatespeech_bucket"] = pd.qcut(
        comment_level["mean_hatespeech"],
        q=3,
        labels=["low", "medium", "high"]
    )

except ValueError:
    print("qcut failed because of duplicate bin edges. Using fixed bins instead.")

    comment_level["hatespeech_bucket"] = pd.cut(
        comment_level["mean_hatespeech"],
        bins=[-0.01, 0.5, 1.5, 2.0],
        labels=["low", "medium", "high"],
        include_lowest=True
    )

comment_level["stratify_label"] = (
    comment_level["target_type"].astype(str)
    + "__"
    + comment_level["hatespeech_bucket"].astype(str)
)

strata_counts = comment_level["stratify_label"].value_counts().sort_index()
annotation_count_summary = comment_level["annotation_count"].describe()

strata_counts

qcut failed because of duplicate bin edges. Using fixed bins instead.


stratify_label
immigrant_only__high           1092
immigrant_only__low            5526
immigrant_only__medium         1385
women_and_immigrant__high        82
women_and_immigrant__low        231
women_and_immigrant__medium      71
women_only__high               2359
women_only__low                6476
women_only__medium             2539
Name: count, dtype: int64

In [22]:
survey_item_summary = comment_level[
    [
        "mean_sentiment",
        "mean_respect",
        "mean_insult",
        "mean_humiliate",
        "mean_status",
        "mean_dehumanize",
        "mean_violence",
        "mean_genocide",
        "mean_attack_defend",
        "mean_hatespeech"
    ]
].describe()

In [24]:
os.makedirs("../results/tables", exist_ok=True)

summary_path = "../results/tables/stratification_summary.txt"

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("STRATIFICATION SUMMARY FOR MAIN EXPERIMENT DATASET\n")
    f.write("=" * 70 + "\n\n")

    f.write("1. DATASET SIZE\n")
    f.write("-" * 70 + "\n")
    f.write(f"Annotation rows: {len(mhs)}\n")
    f.write(f"Unique comments: {mhs['comment_id'].nunique()}\n")
    f.write(f"Unique annotators: {mhs['annotator_id'].nunique()}\n\n")

    f.write("2. TARGET TYPE SUMMARY\n")
    f.write("-" * 70 + "\n")
    f.write(target_summary.to_string())
    f.write("\n\n")

    f.write("3. WOMEN-TARGETED CONTENT: ANNOTATOR GENDER SUMMARY\n")
    f.write("-" * 70 + "\n")
    f.write(women_gender_summary.to_string())
    f.write("\n\n")

    f.write("4. IMMIGRANT-TARGETED CONTENT: ANNOTATOR IDEOLOGY SUMMARY\n")
    f.write("-" * 70 + "\n")
    f.write(immigrant_ideology_summary.to_string())
    f.write("\n\n")

    f.write("5. COMMENT-LEVEL ANNOTATION COUNT SUMMARY\n")
    f.write("-" * 70 + "\n")
    f.write(annotation_count_summary.to_string())
    f.write("\n\n")

    f.write("6. COMMENT-LEVEL SURVEY ITEM MEAN SUMMARY\n")
    f.write("-" * 70 + "\n")
    f.write(survey_item_summary.to_string())
    f.write("\n\n")

    f.write("7. PROPOSED STRATIFICATION LABEL COUNTS\n")
    f.write("-" * 70 + "\n")
    f.write("Stratification label = target_type + hatespeech_bucket\n\n")
    f.write(strata_counts.to_string())
    f.write("\n\n")


print(f"Saved summary to: {summary_path}")

Saved summary to: ../results/tables/stratification_summary.txt
